# A Small Whole Refinery

This notebook strings the `difflow_refinery` units together into a small hydroskimming
refinery, from the crude assay to the finished-product pools:

```text
                    +-> offgas + unstabilised naphtha -> gas plant -+-> fuel gas
                    |                                               +-> LPG
                    |                                               +-> stabilised naphtha
                    |                                                      |
 crude -> CDU ------+                                       naphtha hydrotreater (NHT)
                    |                                          |                |
                    |                                light naphtha    heavy naphtha -> reformer -+-> reformate
                    |                                          |                                 +-> net H2
                    |                                          +--------> gasoline pool <---------+
                    |
                    +-> kerosene + diesel + AGO -> distillate hydrotreater (DHT) -+-> jet pool
                    |                                                             +-> ULSD pool
                    +-> atmospheric residue --------------------------------------> fuel oil pool
```

Every box is a real unit of the plugin, solved rigorously in its own way:

| Unit | Class | Model | How far it is validated |
|---|---|---|---|
| Crude unit (CDU) | `CrudeUnit` | furnace + equation-oriented MESH column, side strippers, pumparounds | against an independent Pyomo/IPOPT column and IDAES property packages |
| Gas plant | `GasCompressor`, `AmineTreater`, `GasPlantColumn` | Peng-Robinson stage columns, O'Connell tray efficiencies | debutanizer against IDAES `TrayColumn` |
| Naphtha and distillate hydrotreaters | `Hydrotreater` (twice) | trickle beds (LHHW HDS by sulfur class, HDN, aromatics saturation), PR separator, H2 recycle, steam stripper | balances and gradients only; **rate constants illustrative** |
| Catalytic reformer | `CatalyticReformer` | three adiabatic beds, equilibrium from Gibbs energies, PR separator, H2 recycle | balances and gradients only; **rate constants illustrative** |
| Product pools | `BlendPool` | Ethyl RT-70 octane, RVP index, mass-basis sulfur, cetane index from composition | rules against published worked examples |

**How to read the numbers.** The balances, the column equations and the equilibrium
thermochemistry are physics. The kinetic constants of the hydrotreaters and the reformer
are *illustrative*: they give the right trends and exact gradients, but the yields, the
product sulfur and the hydrogen numbers are the shape of the answer, not a prediction for
any catalyst. The assay is the made-up test crude used throughout the plugin's examples
(`examples/35`, `38`), given 1.8 wt% sulfur, 1500 wppm nitrogen and 5 wt% CCR.

**Where the API has no connection**, this notebook writes the smallest bridge it needs and
marks it **STAND-IN** in a heading. They are collected in the last section, as the gaps a
whole-refinery model still has.

The run is sized to execute in about a quarter of an hour on four cores: a coarse cut grid
(30 C cuts below 400 C), short gas-plant columns, single-bed naphtha hydrotreater, and one
gradient (of the distillate hydrotreater) rather than one per unit. Most of the time is
compilation: each unit compiles once.

In [1]:
import dataclasses
import time
import warnings

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
from IPython.display import Markdown, display
from tabulate import tabulate

import difflow_refinery as dr
from difflow_refinery import Assay, BlendCharacterization, BlendComponent, BlendPool, column as cc
from difflow_refinery.composition import CompositionRangeWarning
from difflow_refinery.gasplant import (
    AmineTreater, AmineTreaterParams, GasCompressor, GasCompressorParams, GasPlantColumn,
    absorber_deethanizer, debutanizer, fuel_gas, gas_components, lpg_quality)
from difflow_refinery.hydrotreating import Hydrotreater, HydrotreaterParams
from difflow_refinery.reforming import CatalyticReformer, NaphthaFeed, ReformerParams
from difflow_refinery.reforming import products as reformate_products
from difflow_refinery.reforming.thermo import std_liquid_volume

T_START = time.time()
C = 273.15
BBL_D = 86400.0 / cc.BARREL          # m3/s -> bbl/d


def table(rows, headers, floatfmt=".4g"):
    display(Markdown(tabulate(rows, headers, tablefmt="github", floatfmt=floatfmt)))


def tick(label, t0):
    print(f"{label}: {time.time() - t0:.0f} s")

## 1. Crude and crude unit

The crude and the column are those of `examples/38_refinery_gas_plant.ipynb`: 95 000 bbl/d,
a 30-stage atmospheric column with kerosene, diesel and AGO side strippers, two pumparounds,
a 5 % overflash closing the furnace, and a partial condenser held at 40 C so that the
column makes an offgas for the gas plant. The cut grid is coarser than the default (30 C
cuts to 400 C instead of 20 C), which gives 17 pseudo-components and shortens every
compile downstream.

`characterize(..., composition=True)` adds the hydrocarbon types, hydrogen and the sulfur
and nitrogen classes per pseudo-component that the hydrotreaters and the reformer need. The
crude unit characterizes the same assay on the same cut points itself, so its product
streams are on the same component names.

In [2]:
PCT = [0, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
T_C = [-10, 60, 95, 150, 205, 260, 315, 370, 430, 500, 600, 680, 850]
assay = Assay(PCT, [t + C for t in T_C], sg=0.86,
              light_ends={"ethane": 0.05, "propane": 0.5, "isobutane": 0.3,
                          "n_butane": 1.0, "isopentane": 0.8, "n_pentane": 1.5},
              sulfur_wt=1.8, nitrogen_wppm=1500.0, ccr_wt=5.0)
cut_points = dr.default_cut_points(assay, ((673.15, 30.0), (873.15, 60.0), (np.inf, 150.0)))
with warnings.catch_warnings():
    warnings.simplefilter("ignore", CompositionRangeWarning)
    char = dr.characterize(assay, cut_points, composition=True)

BPD = 95_000.0
Vf = BPD * cc.BARREL / 86400.0
cdu_params = cc.CrudeColumnParams(
    n_stages=30, feed_stage=27, P_top=1.5e5, P_bottom=1.9e5, P_condenser=1.3e5,
    bottom_steam=150.0, steam_T=C + 260.0, condenser="partial",
    side_products=(cc.SideProduct("kero", 9, 4, steam=40.0),
                   cc.SideProduct("diesel", 16, 4, steam=40.0),
                   cc.SideProduct("ago", 22, 3, steam=20.0)),
    pumparounds=(cc.Pumparound("pa1", 12, 10), cc.Pumparound("pa2", 19, 17)),
    specs=(cc.product_rate("naphtha", 0.20 * Vf), cc.product_rate("kero", 0.11 * Vf),
           cc.product_rate("diesel", 0.17 * Vf), cc.product_rate("ago", 0.05 * Vf),
           cc.pumparound_duty("pa1", 15e6), cc.pumparound_delta_t("pa1", 60.0),
           cc.pumparound_duty("pa2", 20e6), cc.pumparound_delta_t("pa2", 60.0),
           cc.overflash(0.05), cc.stage_temperature(0, C + 40.0)),
)
t0 = time.time()
cdu_unit = dr.CrudeUnit(assay, cdu_params, cut_points=cut_points)
assert tuple(cdu_unit.crude.names) == tuple(char.names)
cdu = cdu_unit.solve(BPD, T=C + 240.0, P=6e5)
tick(f"CDU converged: {bool(cdu.converged)}", t0)
print(cdu.table())
P = {name: {k: v for k, v in s.items()} for name, s in cdu.products.items()}

CDU converged: True: 56 s
product        bbl/d  vol %   wt %    API  TBP5 C  TBP50 C  TBP95 C
offgas           136    0.1    0.1  122.3     -89      -10       89
naphtha        19000   20.0   16.7   65.4      -5       96      162
kero           10450   11.0   10.2   46.7     126      183      236
diesel         16150   17.0   16.5   37.6     195      260      326
ago             4750    5.0    5.1   30.6     271      333      383
residue        44514   46.9   51.4   18.4     322      476      759

coil outlet 313.9 C, 69.7 mol% vaporised; furnace 42.8 MW absorbed, 50.3 MW fired
condenser 31.9 MW, pumparounds 15.0, 20.0 MW


## 2. Gas plant

The offgas and the unstabilised naphtha go to the saturated gas plant exactly as in
`examples/38_refinery_gas_plant.ipynb`, which is the documented path from the crude unit:
a two-stage compressor to 14.5 bar, an amine treater on the compressed vapour, an
absorber-deethanizer with the whole naphtha as lean oil (overhead: fuel gas), and a
debutanizer (overhead: LPG; bottoms: stabilised naphtha). The columns are shorter than in
example 38 (12 and 16 trays) to save compile time.

### STAND-IN: crude-unit streams into the gas plant's component table

There is no adapter from a crude-unit product to a `gas_components` table. The bridge is
example 38's: keep the cuts that carry more than 0.1 % of the naphtha, fold the rest
(about 1e-4 of it) into the heaviest kept cut, drop the water the accumulator decants.
And because the crude unit does not make H2S (the assay's sulfur stays on the cuts), the
offgas is given an **assumed** 2 mol % H2S so the amine has something to remove.

In [3]:
off = {k[2:]: float(v) for k, v in P["offgas"].items() if k.startswith("F_")}
nap = {k[2:]: float(v) for k, v in P["naphtha"].items() if k.startswith("F_")}
tot = sum(nap.values())
gp_cuts = [n for n in char.pseudo_names if nap.get(n, 0.0) / tot > 1e-3]
folded = [n for n in char.pseudo_names if n not in gp_cuts]
LIGHT = ["hydrogen_sulfide", "ethane", "propane", "isobutane", "n_butane", "isopentane", "n_pentane"]
comps = gas_components(LIGHT, pseudo=char, cuts=gp_cuts)


def gp_stream(flows, T, P):
    s = {f"F_{n}": 0.0 for n in comps.names}
    for k, v in flows.items():
        if k != "water":
            s[f"F_{gp_cuts[-1] if k in folded else k}"] += v
    s.update(T=T, P=P)
    return s


def gp_flows(s):
    return jnp.stack([jnp.asarray(s[f"F_{n}"]) for n in comps.names])


def gp_mass(s):                                      # kg/s
    return float(jnp.sum(gp_flows(s) * comps.MW)) / 1000.0


H2S = 0.02 * sum(v for k, v in off.items() if k != "water")      # ASSUMED, see above
offgas = gp_stream(dict(off, hydrogen_sulfide=H2S), C + 40.0, 1.3e5)
naphtha = gp_stream(nap, C + 40.0, 1.3e5)

t0 = time.time()
comp_gas, condensate, ci = GasCompressor(GasCompressorParams(comps, outlet_P=14.5e5, n_stages=2))(offgas)
sweet, acid, ai = AmineTreater(AmineTreaterParams())(comp_gas)
gp_feed = {k: sweet[k] + condensate[k] for k in sweet if k.startswith("F_")}
gp_feed.update(T=C + 40.0, P=14.5e5)
deeth = GasPlantColumn(absorber_deethanizer(comps, n_trays=12, feed_tray=4, c2_in_bottoms=0.002))
fgas, deeth_btms, di = deeth(dict(naphtha, P=14.5e5), gp_feed)
tick(f"deethanizer converged: {bool(di['converged'])}", t0)
t0 = time.time()
debut = GasPlantColumn(debutanizer(comps, n_trays=16, feed_tray=8, c5_in_lpg=0.01, c4_in_naphtha=0.01))
lpg, stab, bi = debut(deeth_btms)
tick(f"debutanizer converged: {bool(bi['converged'])}", t0)

od, ob = di["outputs"], bi["outputs"]
q = lpg_quality(gp_flows(lpg), comps, grade="commercial_propane")
fg = fuel_gas(gp_flows(fgas), comps)
table([("fuel gas (kg/s, LHV MJ/kg)", f"{gp_mass(fgas):.3f}, {float(fg['lhv_mass']) / 1e6:.1f}"),
       ("LPG (kg/s); propane / butanes+ (mol frac)",
        f"{gp_mass(lpg):.2f}; {float(q['values']['propane']):.2f} / {float(q['values']['butanes_plus']):.2f}"),
       ("stabilised naphtha (kg/s), RVP (kPa)", f"{gp_mass(stab):.2f}, {float(ob['bottoms.rvp']) / 1e3:.1f}"),
       ("compressor (MW)", f"{float(ci['power']) / 1e6:.3f}"),
       ("deethanizer / debutanizer reboiler (MW)",
        f"{float(od['reboiler.duty']) / 1e6:.2f} / {float(ob['reboiler.duty']) / 1e6:.2f}")],
      ["gas plant", "value"])
into = gp_flows(offgas) + gp_flows(naphtha)
out = sum(gp_flows(s) for s in (fgas, acid, lpg, stab))
print(f"largest component imbalance across the gas plant: {float(jnp.max(jnp.abs(out - into))):.1e} mol/s")

deethanizer converged: True: 65 s


debutanizer converged: True: 47 s


| gas plant                                 | value             |
|-------------------------------------------|-------------------|
| fuel gas (kg/s, LHV MJ/kg)                | 0.021, 46.8       |
| LPG (kg/s); propane / butanes+ (mol frac) | 1.65; 0.32 / 0.66 |
| stabilised naphtha (kg/s), RVP (kPa)      | 23.56, 32.3       |
| compressor (MW)                           | 0.017             |
| deethanizer / debutanizer reboiler (MW)   | 8.41 / 11.70      |

largest component imbalance across the gas plant: 2.8e-13 mol/s


## 3. Naphtha hydrotreater

The stabilised naphtha goes to a `Hydrotreater` set up as a naphtha unit: one bed, 300 C
inlet, LHSV 0.7 h-1, 400 Nm3/m3 of treat gas, a cooler stripper. The `Hydrotreater` was
built and tested on diesel and kerosene; this is its first use on a naphtha, and its rate
constants were chosen for a diesel (illustrative, see the module docs). That shows:

- At a naphtha unit's usual 30 bar and 330 C, with these constants the unit desulfurizes
  only to about 300 wppm, and the aromatics-saturation equilibrium runs *backwards*: at
  that hydrogen partial pressure it dehydrogenates naphthenes, the beds cool, and the
  chemical hydrogen consumption comes out negative. That is the equilibrium layer doing what
  thermodynamics says at that pressure; a real CoMo catalyst does not dehydrogenate at
  that rate, which is a property of the illustrative constants.
- The conditions here (60 bar, LHSV 0.7, 400 Nm3/m3) are therefore harsher than a real
  naphtha hydrotreater's, and are what it takes to get a few wppm. A reformer wants below
  about 0.5 wppm; these kinetics do not get there at any condition tried, and the reformer
  model carries no sulfur, so nothing downstream notices.
- A sweep of the conditions (not run here) is cheap once the unit is compiled: a re-solve
  with new numbers takes about a second. One point of it, 320 C / 50 bar / LHSV 0.5, does
  not return `converged=False` but stops with a NaN inside a linear solve.

The gas plant's stabilised naphtha is on `comps`, whose cuts are a subset of `char.names`,
so it goes into the hydrotreater as it is (the 6e-8 mol/s of H2S it still carries is not a
characterization component and is dropped).

**Reporting quirk.** The unit's `gas.yield` output subtracts every light end in the feed,
including the pentanes, which then leave in the liquid products, so on a naphtha feed it
goes negative. The refinery mass balance in section 8 is computed from the streams instead.

In [4]:
nht_feed = {k: float(v) for k, v in stab.items() if k.startswith("F_") and k[2:] in char.names}
nht_feed.update(T=C + 40.0, P=3e5)
nht_params = HydrotreaterParams(T_in=(C + 300.0,), quench=None, bed_fractions=(1.0,), P=60e5,
                                lhsv=0.7, h2_oil=400.0, stripper_feed_T=C + 150.0)
t0 = time.time()
nht = Hydrotreater(char, nht_feed, nht_params)
nht_res = nht.solve(nht_feed)
tick(f"naphtha hydrotreater converged: {bool(nht_res.converged)}", t0)
print(nht_res.table())
print("closures:", {k: f"{float(v):.0e}" for k, v in nht_res.balances.items()})

naphtha hydrotreater converged: True: 99 s
WABT 308.9 C, bed dT 13.3 K
product: S 4.9 wppm, N 17.7 wppm, SG 0.7476, aromatics 17.2 vol%, cetane index 28.5
yields (mass): product 82.45 %, wild naphtha 16.59 %, gas -10.239 %
H2: chemical 11.1 Nm3/m3 (66 scf/bbl), makeup 39.8 Nm3/m3, purity 97.2 %
recycle compressor 98.5 kW, purge 27.997 mol/s
closures: {'C': '0e+00', 'H': '8e-16', 'N': '2e-16', 'S': '4e-16', 'mass': '1e-16'}


### STAND-IN: naphtha splitter after the hydrotreater

A refinery splits the treated naphtha: the light naphtha (C5s and the first cut, which holds
the benzene precursors methylcyclopentane and cyclohexane) goes to gasoline or an
isomerization unit, and only the heavy naphtha is reformed. The gas plant has a rigorous
`splitter` factory, but the treated naphtha leaves the hydrotreater on the hydrotreater's
product grid (`res.product_char`), not on a `gas_components` table, and there is no adapter
between the two. Rather than a third column compile, the split here is a **sharp
pseudo-component split** at 85 C TBP of the hydrotreater's product plus its wild naphtha:
every component below goes to light naphtha, every one above to the reformer.

In [5]:
def cut_split(stream, char_, T_cut):
    # Sharp split of an F_<name> stream at a TBP temperature: (below, above). STAND-IN.
    Tb = dict(zip(char_.names, np.asarray(char_.Tb)))
    lo = {k: v * (1.0 if Tb[k[2:]] < T_cut else 0.0) for k, v in stream.items() if k.startswith("F_")}
    hi = {k: v - lo[k] for k, v in stream.items() if k.startswith("F_")}
    for s in (lo, hi):
        s.update(T=stream["T"], P=stream["P"])
    return lo, hi


nht_char = nht_res.product_char
treated = {k: nht_res.product_stream("product")[k] + nht_res.product_stream("wild_naphtha")[k]
           for k in nht_res.product_stream("product") if k.startswith("F_")}
treated.update(T=298.15, P=101325.0)
light_naphtha, heavy_naphtha = cut_split(treated, nht_char, C + 85.0)
lsr = BlendComponent.from_stream("light naphtha", light_naphtha, nht_char)
print(f"light naphtha {float(lsr.available_volume) * BBL_D:,.0f} bbl/d, "
      f"heavy naphtha {float(BlendComponent.from_stream('h', heavy_naphtha, nht_char).available_volume) * BBL_D:,.0f} bbl/d")

light naphtha 6,884 bbl/d, heavy naphtha 10,455 bbl/d


## 4. Catalytic reformer

### STAND-IN: hydrotreated naphtha into a `NaphthaFeed`

`NaphthaFeed.from_characterization(char, flows)` is the documented path from a
characterized naphtha to the reformer's P/N/A lumps by carbon number, and the
hydrotreater's product names are `char.names`. But the mapping reads the hydrocarbon types
from `char.composition`, the **untreated** crude's, so what the hydrotreater did to the
cuts (saturated a little aromatic, removed the sulfur and nitrogen) does not reach the
reformer. On a straight-run naphtha at these conditions that is a small error (the
treated product's aromatics are within a few tenths of a percent of the feed's), and the
reformer carries no sulfur anyway; it is still a gap. The default iso/normal and
MCP/cyclohexane splits of `from_characterization` are illustrative.

In [6]:
ref_feed = NaphthaFeed.from_characterization(char, heavy_naphtha)
g = ref_feed.group_fractions("volume")
print(f"reformer feed {float(ref_feed.mass_flow):.2f} kg/s: P/N/A {100 * float(g['paraffins']):.0f}/"
      f"{100 * float(g['naphthenes']):.0f}/{100 * float(g['aromatics']):.0f} vol%, "
      f"N+2A {float(ref_feed.n_plus_2a()):.0f}")
t0 = time.time()
reformer = CatalyticReformer(ReformerParams())          # WAIT 500 C, 12 bar, H2/HC 5, LHSV 1.5
ref = reformer.solve(ref_feed)
tick(f"reformer converged: {bool(ref.converged)} in {ref.iterations} recycle iterations", t0)
print(ref.summary())
print("closures:", {k: f"{float(v):.0e}" for k, v in ref.balances().items()})
ro = ref.outputs()

reformer feed 14.55 kg/s: P/N/A 56/24/21 vol%, N+2A 65


reformer converged: True in 25 recycle iterations: 51 s


WAIT 500.0 C, WABT 473.4 C, P_sep 12.0 bar, H2/HC 5.00
reactor dT (K): -64.6, -59.8, -46.5
reformate: 81.8 vol% (81.7 C5+), RON 97.7 (linear 89.1), MON 87.5, aromatics 71.0 vol%, benzene 1.54 vol%, RVP 2.0 psi
net H2 2.59 wt% of feed at 88.3 mol%; LPG 0.385 kg/s, fuel gas 0.040 kg/s
fired 51.13 MW, compressor 0.599 MW, coke 0.69 kg/h, cycle 291 d
closures: {'mass': '-5e-13', 'carbon': '-5e-13', 'hydrogen': '-4e-13', 'energy': '1e-13', 'adiabatic_1': '-3e-16', 'adiabatic_2': '-4e-15', 'adiabatic_3': '7e-15'}


The reformer's fired duty is large for its size because the model has no feed-effluent
exchanger: the charge heater does all the work of heating naphtha and recycle gas from the
separator to the first reactor (see the reformer docs).

## 5. Distillate hydrotreater

Kerosene, diesel and AGO are combined and hydrotreated together: two beds with a 15 %
quench, 340 C first-bed inlet, 60 bar, LHSV 1 h-1, 300 Nm3/m3. The crude-unit products
(stripping water and light-end traces included) go in as they are, which the tests cover.

In [7]:
dht_feed = {}
for name in ("kero", "diesel", "ago"):
    for k, v in P[name].items():
        if k.startswith("F_"):
            dht_feed[k] = dht_feed.get(k, 0.0) + float(v)
dht_feed.update(T=C + 200.0, P=3e5)
dht_params = HydrotreaterParams(T_in=(C + 340.0,), P=60e5, lhsv=1.0, h2_oil=300.0)
t0 = time.time()
dht = Hydrotreater(char, dht_feed, dht_params)
dht_res = dht.solve(dht_feed)
tick(f"distillate hydrotreater converged: {bool(dht_res.converged)}", t0)
print(dht_res.table())
print("closures:", {k: f"{float(v):.0e}" for k, v in dht_res.balances.items()})

distillate hydrotreater converged: True: 113 s
WABT 350.9 C, bed dT 11.8, 8.1 K
product: S 8.3 wppm, N 96.5 wppm, SG 0.8212, aromatics 18.5 vol%, cetane index 51.3
yields (mass): product 96.73 %, wild naphtha 2.87 %, gas 0.697 %
H2: chemical 31.1 Nm3/m3 (184 scf/bbl), makeup 50.3 Nm3/m3, purity 96.0 %
recycle compressor 123.9 kW, purge 35.221 mol/s
closures: {'C': '1e-16', 'H': '0e+00', 'N': '5e-16', 'S': '5e-16', 'mass': '0e+00'}


### STAND-IN: splitting the distillate hydrotreater's product into jet and diesel

The `Hydrotreater` has one liquid product. A refinery runs a kerosene and a diesel
hydrotreater, or fractionates a combined product; the plugin has neither the fractionator
nor a jet/diesel splitter on the hydrotreater's product grid. The same sharp
pseudo-component split is used, at 240 C TBP. The distillate hydrotreater's wild naphtha
from the stripper goes to the gasoline pool as it is, rather than being reprocessed.

In [8]:
dht_char = dht_res.product_char
JET_CUT = C + 240.0


def jet_and_diesel(res):
    return cut_split(res.product_stream("product"), res.product_char, JET_CUT)


jet_stream, diesel_stream = jet_and_diesel(dht_res)

## 6. Product pools

Each pool uses `BlendPool`'s default specs (illustrative: US regular summer gasoline,
Jet A, ULSD S15, a 380 cSt fuel oil). The recipe is "everything the refinery makes of that
product goes into its pool", so the margins say whether this refinery's own streams make
spec without purchased blendstocks.

### STAND-IN: properties nothing in the plugin computes

- **Straight-run octane.** There is no straight-run octane correlation (out of scope in the
  blend-pool docs), so the light naphtha's RON/MON are **assumed** (70/68), and the
  distillate hydrotreater's heavier wild naphtha's (60/58). The light naphtha's benzene is
  taken as its aromatics. Gravity, sulfur and RVP come from their compositions.
- **Reformate sulfur.** The reformer does not carry sulfur; the reformate is given the
  naphtha hydrotreater's product sulfur (an upper bound: the reformer's H2S goes to the
  net gas).
- **Flash, freeze and smoke points, viscosity.** None is computed (the hydrotreater docs
  list smoke point as not done). Jet takes **assumed** flash 42 C, freeze -47 C and smoke
  22 mm; diesel an assumed flash of 60 C; fuel oil an assumed viscosity of 300 cSt.

One pool takes one mode, so the gasoline pool is in property mode (the reformate is a
property-mode component) and the light naphtha's composition-derived properties are
copied into a property-mode component.

In [9]:
reformate = reformate_products.blend_component(
    "reformate", ref.flows("reformate"), S_ppm=nht_res.outputs["product.S_wppm"])
V_ref = std_liquid_volume(ref.flows("reformate"))
lsr_props = {k: lsr.properties[k] for k in ("SG", "RVP_psi", "S_ppm", "aromatics_vol",
                                            "naphthenes_vol", "paraffins_vol", "olefins_vol")}
light = BlendComponent.from_properties("light naphtha", RON=70.0, MON=68.0,          # ASSUMED
                                       benzene_vol=lsr_props["aromatics_vol"], **lsr_props)
dht_wild = dht_res.product_stream("wild_naphtha")
wn = BlendComponent.from_stream("DHT wild naphtha", dht_wild, dht_char)
wild = BlendComponent.from_properties("DHT wild naphtha", RON=60.0, MON=58.0,              # ASSUMED
                                      benzene_vol=0.0,
                                      **{k: wn.properties[k] for k in lsr_props})
gasoline = BlendPool("gasoline")([reformate, light, wild],
                                 [V_ref, lsr.available_volume, wn.available_volume], basis="volume_flow")

jet_c = BlendComponent.from_stream("jet", jet_stream, dht_char,
                                   flash_C=42.0, freeze_C=-47.0, smoke_mm=22.0)      # ASSUMED
jet = BlendPool("jet")([jet_c], [1.0], basis="split")
diesel_c = BlendComponent.from_stream("diesel", diesel_stream, dht_char, flash_C=60.0)  # ASSUMED
ulsd = BlendPool("ulsd")([diesel_c], [1.0], basis="split")

fo_char = BlendCharacterization.from_characterization(char, contaminants=True)
resid_c = BlendComponent.from_stream("residue", P["residue"], fo_char, viscosity_cSt=300.0)  # ASSUMED
fuel_oil = BlendPool("fuel_oil")([resid_c], [1.0], basis="split")

for name, r in (("gasoline", gasoline), ("jet", jet), ("ULSD", ulsd), ("fuel oil", fuel_oil)):
    print(f"\n{name}: {float(r.volume) * BBL_D:,.0f} bbl/d")
    table([(s, f"{float(m):+.4g}", "on spec" if float(m) >= 0 else "OFF SPEC") for s, m in r.margins.items()],
          ["spec", "margin", ""])


gasoline: 16,355 bbl/d


| spec         |   margin |          |
|--------------|----------|----------|
| RON >= 91    |   -5.195 | OFF SPEC |
| MON >= 82    |   -2.976 | OFF SPEC |
| RVP_psi <= 9 |    3.85  | on spec  |
| S_ppm <= 10  |    3.745 | on spec  |


jet: 14,589 bbl/d


| spec            |   margin |         |
|-----------------|----------|---------|
| S_ppm <= 3000   |    +2996 | on spec |
| freeze_C <= -40 |       +7 | on spec |
| smoke_mm >= 18  |       +4 | on spec |
| flash_C >= 38   |       +4 | on spec |


ULSD: 15,978 bbl/d


| spec               |   margin |         |
|--------------------|----------|---------|
| S_ppm <= 15        |    2.764 | on spec |
| cetane_index >= 40 |   17.31  | on spec |
| flash_C >= 52      |    8     | on spec |
| T90_d86_C <= 338   |    6.838 | on spec |


fuel oil: 44,510 bbl/d


| spec                 |     margin |          |
|----------------------|------------|----------|
| S_ppm <= 5000        | -2.559e+04 | OFF SPEC |
| viscosity_cSt <= 380 | 80         | on spec  |
| SG <= 0.991          |  0.04705   | on spec  |
| CCR_wt <= 18         |  8.299     | on spec  |

What the pools say about this refinery:

- **Gasoline is short of octane** (RON about 86 against 91). The reformate is at RON
  98, but nearly half of the pool is straight-run naphtha (the light naphtha and the
  distillate hydrotreater's wild naphtha) at an assumed RON of 70 and 60.
  A hydroskimming refinery fixes that with a C5/C6 isomerization unit on the light naphtha
  (`IsomerizationUnit`, `examples/39_refinery_isomerization.ipynb`), a higher reformer
  severity, or bought octane. The sulfur spec is met because the naphtha hydrotreater was
  run hard (above).
- **Jet and ULSD make spec**, the ULSD with a few ppm of sulfur to spare. The jet's flash,
  freeze and smoke margins only restate the assumed values.
- **The fuel oil is far off its sulfur spec**: the atmospheric residue carries most of the
  crude's sulfur (see the sulfur table below), at about 3 wt% against 0.5. Nothing in this
  refinery treats residue; that is what a vacuum unit and a cracker or a residue
  desulfurizer are for.

## 7. The hydrogen balance

The reformer is the refinery's hydrogen plant; the two hydrotreaters are its consumers. The
hydrotreaters' makeup is what they take from the header: chemical consumption plus what
dissolves in the separator liquid and what goes out with the purge.

### STAND-IN: the hydrogen header

There is no hydrogen-network block, so the balance below is arithmetic on the units'
outputs. It is not a closed loop: the hydrotreaters were solved with their default makeup
gas (97 mol % H2, 3 % CH4), while the reformer's net gas is at the purity it reports. A
header that delivered the reformer's gas to the hydrotreaters would lower their hydrogen
partial pressure, and would need a PSA (or a purity spec on the makeup) to be closed.

In [10]:
MOL_PER_NM3 = 101325.0 / (8.314462618 * 273.15)
h2_make = float(ro["h2.net_mol_s"])
rows = [("reformer net H2", h2_make, f"{float(ro['h2.purity']):.1f}", "")]
for name, r in (("naphtha HDT", nht_res), ("distillate HDT", dht_res)):
    o = r.outputs
    rows.append((f"{name} makeup H2", -float(o["h2.makeup"]), "97.0 (assumed makeup)",
                 f"chemical {float(o['h2.chemical']):.1f}, dissolved {float(o['h2.dissolved']):.1f}, "
                 f"purge {float(o['h2.purge']):.1f}"))
surplus = h2_make - sum(float(r.outputs["h2.makeup"]) for r in (nht_res, dht_res))
rows.append(("surplus to fuel or export", surplus, "", f"{surplus / MOL_PER_NM3 * 3600 / 1e3:.1f} kNm3/h"))
table(rows, ["hydrogen", "mol/s", "purity (mol %)", "where it goes (mol/s)"], floatfmt=".1f")

| hydrogen                  |   mol/s | purity (mol %)        | where it goes (mol/s)                     |
|---------------------------|---------|-----------------------|-------------------------------------------|
| reformer net H2           |   186.5 | 88.3                  |                                           |
| naphtha HDT makeup H2     |   -57.2 | 97.0 (assumed makeup) | chemical 15.9, dissolved 14.0, purge 27.2 |
| distillate HDT makeup H2  |  -129.4 | 97.0 (assumed makeup) | chemical 79.9, dissolved 15.7, purge 33.8 |
| surplus to fuel or export |    -0.1 |                       | -0.0 kNm3/h                               |

On these numbers the reformer's net hydrogen just covers the two hydrotreaters' makeup.
That is a coincidence of the conditions chosen, not a design: the naphtha hydrotreater's
severity, and the distillate hydrotreater's purge, each move it by tens of mol/s. Most of
the makeup is not chemistry: the purges and the hydrogen dissolved in the separator
liquids are about half of it.

## 8. The refinery in one table

Product rates, the qualities the specs are written on, and the energy the units report.
"Fuel gas" collects the gas plant's deethanizer overhead, the reformer's fuel gas, and the
hydrotreaters' purge and stripper off-gas (all as reported, untreated). "LPG" adds the
reformer's LPG to the gas plant's.

Then two checks that the stand-in connections lose nothing: the refinery's overall mass
balance over every terminal stream (water and steam included; each unit closes its own to
round-off, so this tests the bridges), and where the crude's sulfur ends up.

The energy table has no hydrotreater furnaces: the `Hydrotreater` takes its bed-inlet
temperature as a spec and does not model the charge heater.

In [11]:
def flows_mass(res, unit, *names):
    return sum(float(res.streams[n].mass(unit.layout)) for n in names)


fuel_kg_s = (gp_mass(fgas) + float(ro["fuel_gas.kg_s"])
             + flows_mass(nht_res, nht, "purge", "off_gas") + flows_mass(dht_res, dht, "purge", "off_gas"))
lpg_kg_s = gp_mass(lpg) + float(ro["lpg.kg_s"])
t_h = lambda kg_s: kg_s * 3.6
products = [
    ("fuel gas", "", t_h(fuel_kg_s), ""),
    ("LPG", "", t_h(lpg_kg_s), f"gas-plant LPG vapour pressure margin {float(q['margins']['vapor_pressure']) / 1e3:.0f} kPa"),
    ("gasoline", float(gasoline.volume) * BBL_D, t_h(float(gasoline.mass)),
     f"RON {float(gasoline.properties['RON']):.1f}, MON {float(gasoline.properties['MON']):.1f}, "
     f"RVP {float(gasoline.properties['RVP_psi']):.1f} psi, S {float(gasoline.properties['S_ppm']):.2g} ppm"),
    ("jet", float(jet.volume) * BBL_D, t_h(float(jet.mass)),
     f"S {float(jet.properties['S_ppm']):.2g} ppm, SG {float(jet.properties['SG']):.3f}"),
    ("ULSD", float(ulsd.volume) * BBL_D, t_h(float(ulsd.mass)),
     f"S {float(ulsd.properties['S_ppm']):.2g} ppm, cetane index {float(ulsd.properties['cetane_index']):.1f}, "
     f"T90 {float(ulsd.properties['T90_d86_C']):.0f} C"),
    ("fuel oil", float(fuel_oil.volume) * BBL_D, t_h(float(fuel_oil.mass)),
     f"S {float(fuel_oil.properties['S_ppm']) / 1e4:.2f} wt%, SG {float(fuel_oil.properties['SG']):.3f}, "
     f"CCR {float(fuel_oil.properties['CCR_wt']):.1f} wt%"),
    ("H2 surplus", "", t_h(surplus * 2.01588e-3), ""),
]
table(products, ["product", "bbl/d", "t/h", "key qualities"], floatfmt=",.1f")
# Overall mass balance over the terminal streams (water and steam included).
W = 18.01528e-3
crude_feed = cdu_unit.feed(BPD, T=C + 240.0, P=6e5)
crude_kg_s = sum(float(crude_feed[f"F_{n}"]) * mw for n, mw in zip(char.names, np.asarray(char.component_MW))) / 1000.0
cdu_steam = (cdu_params.bottom_steam + sum(sp.steam for sp in cdu_params.side_products)) * W
cdu_water = sum(float(P[n].get("F_water", 0.0)) for n in ("water", "offgas", "naphtha", "residue")) * W


def char_mass(stream, bchar):
    return float(sum(stream[f"F_{n}"] * m for n, m in zip(bchar.names, np.asarray(bchar.mw)))) / 1000.0


HDT_IN, HDT_OUT = ("makeup", "steam"), ("off_gas", "purge", "acid_gas", "hps_water", "sour_water")
mass_in = {"crude": crude_kg_s, "CDU stripping steam": cdu_steam, "assumed offgas H2S": H2S * 34.081e-3,
           "HDT makeup gas and steam": flows_mass(nht_res, nht, *HDT_IN) + flows_mass(dht_res, dht, *HDT_IN)}
mass_out = {"gas plant fuel gas, acid gas, LPG": gp_mass(fgas) + gp_mass(acid) + gp_mass(lpg),
            "light naphtha + DHT wild naphtha": char_mass(light_naphtha, nht_char) + char_mass(dht_wild, dht_char),
            "reformate, LPG, fuel gas, net gas": sum(float(ro[k]) for k in ("reformate.kg_s", "lpg.kg_s",
                                                                          "fuel_gas.kg_s", "net_gas.kg_s")),
            "jet + diesel": char_mass(jet_stream, dht_char) + char_mass(diesel_stream, dht_char),
            "HDT gases and waters": flows_mass(nht_res, nht, *HDT_OUT) + flows_mass(dht_res, dht, *HDT_OUT),
            "residue": char_mass(P["residue"], fo_char),
            "CDU water (decanted and in products)": cdu_water}
m_in, m_out = sum(mass_in.values()), sum(mass_out.values())
table([(k, t_h(v)) for k, v in mass_in.items()] + [("**in**", t_h(m_in))]
      + [(k, t_h(v)) for k, v in mass_out.items()] + [("**out**", t_h(m_out))],
      ["refinery mass balance", "t/h"], floatfmt=",.3f")
print(f"imbalance {(m_out - m_in) / m_in:+.1e} of the inputs ({t_h(m_out - m_in):+.3f} t/h), of which:")
dropped = {name: flows_mass(r, u, "product", "wild_naphtha")
           - char_mass(r.product_stream("product"), r.product_char)
           - char_mass(r.product_stream("wild_naphtha"), r.product_char)
           for name, r, u in (("naphtha HDT", nht_res, nht), ("distillate HDT", dht_res, dht))}
for name, v in dropped.items():
    print(f"  {name}: gas dissolved in its liquids that product_stream() leaves out: {t_h(-v):+.3f} t/h")
print(f"  reformer feed against the heavy naphtha it was mapped from: "
      f"{t_h(float(ref_feed.mass_flow) - char_mass(heavy_naphtha, nht_char)):+.3f} t/h")

# Where the crude's sulfur goes (the assumed offgas H2S is left out: it is not the assay's).
crude_S = crude_kg_s * 0.018


def hdt_S(r):
    o = r.outputs
    feed = float(o["feed.S_wppm"] * o["feed.rate"]) * 1e-6
    liquid = float(o["product.S_wppm"] * o["product.rate"] + o["naphtha.S_wppm"] * o["naphtha.rate"]) * 1e-6
    return feed, liquid


nS_feed, nS_liq = hdt_S(nht_res)
dS_feed, dS_liq = hdt_S(dht_res)
fo_S = float(fuel_oil.properties["S_ppm"]) * 1e-6 * char_mass(P["residue"], fo_char)
s_rows = [("fuel oil (the atmospheric residue)", fo_S),
          ("naphtha HDT liquids (to reformer and gasoline)", nS_liq),
          ("distillate HDT liquids (jet, ULSD, wild naphtha)", dS_liq),
          ("H2S from both hydrotreaters (to sulfur recovery)", nS_feed - nS_liq + dS_feed - dS_liq),
          ("not carried by the gas plant's cuts", crude_S - fo_S - nS_feed - dS_feed)]
table([(k, 3.6 * v, 100 * v / crude_S) for k, v in s_rows],
      [f"crude sulfur, {3.6 * crude_S:.2f} t/h", "t/h", "% of crude S"], floatfmt=".3f")

energy = [
    ("CDU furnace", float(cdu.column.furnace_fired_duty) / 1e6, "fired"),
    ("reformer heaters", float(ro["heaters.fired_MW"]), "fired"),
    ("gas-plant reboilers", float(od["reboiler.duty"] + ob["reboiler.duty"]) / 1e6, "absorbed (not fired)"),
    ("compressors (gas plant, reformer, 2 x HDT)",
     (float(ci["power"]) + float(nht_res.outputs["compressor.power"]) + float(dht_res.outputs["compressor.power"])) / 1e6
     + float(ro["compressor.power_MW"]), "shaft"),
]
table(energy, ["energy", "MW", ""], floatfmt=".1f")

| product    |    bbl/d |   t/h | key qualities                                |
|------------|----------|-------|----------------------------------------------|
| fuel gas   |          |   2.5 |                                              |
| LPG        |          |   7.3 | gas-plant LPG vapour pressure margin 812 kPa |
| gasoline   | 16,355.2 |  82.6 | RON 85.8, MON 79.0, RVP 5.1 psi, S 6.3 ppm   |
| jet        | 14,589.2 |  76.9 | S 3.7 ppm, SG 0.796                          |
| ULSD       | 15,978.2 |  89.3 | S 12 ppm, cetane index 57.3, T90 331 C       |
| fuel oil   | 44,509.8 | 278.1 | S 3.06 wt%, SG 0.944, CCR 9.7 wt%            |
| H2 surplus |          |  -0.0 |                                              |

| refinery mass balance                |     t/h |
|--------------------------------------|---------|
| crude                                | 540.687 |
| CDU stripping steam                  |  16.214 |
| assumed offgas H2S                   |   0.006 |
| HDT makeup gas and steam             |   4.256 |
| **in**                               | 561.163 |
| gas plant fuel gas, acid gas, LPG    |   6.029 |
| light naphtha + DHT wild naphtha     |  36.545 |
| reformate, LPG, fuel gas, net gas    |  52.367 |
| jet + diesel                         | 166.147 |
| HDT gases and waters                 |   5.691 |
| residue                              | 278.076 |
| CDU water (decanted and in products) |  16.214 |
| **out**                              | 561.070 |

imbalance -1.7e-04 of the inputs (-0.093 t/h), of which:
  naphtha HDT: gas dissolved in its liquids that product_stream() leaves out: -0.022 t/h
  distillate HDT: gas dissolved in its liquids that product_stream() leaves out: -0.047 t/h
  reformer feed against the heavy naphtha it was mapped from: -0.022 t/h


| crude sulfur, 9.73 t/h                           |   t/h |   % of crude S |
|--------------------------------------------------|-------|----------------|
| fuel oil (the atmospheric residue)               | 8.507 |         87.408 |
| naphtha HDT liquids (to reformer and gasoline)   | 0.000 |          0.004 |
| distillate HDT liquids (jet, ULSD, wild naphtha) | 0.001 |          0.014 |
| H2S from both hydrotreaters (to sulfur recovery) | 1.224 |         12.574 |
| not carried by the gas plant's cuts              | 0.000 |          0.000 |

| energy                                     |   MW |                      |
|--------------------------------------------|------|----------------------|
| CDU furnace                                | 50.3 | fired                |
| reformer heaters                           | 51.1 | fired                |
| gas-plant reboilers                        | 20.1 | absorbed (not fired) |
| compressors (gas plant, reformer, 2 x HDT) |  0.8 | shaft                |

The overall balance closes to about 1e-4, not to round-off, and the lines under it say
why: both are the stand-in bridges, not the units.

- `Hydrotreater.product_stream()` keeps only the components a blend pool can take (C3+
  light ends and the cuts), so the hydrogen, H2S, methane and ethane still dissolved in
  the stripper bottoms and the wild naphtha are not passed on.
- `NaphthaFeed.from_characterization` maps the treated cuts with the *untreated* crude's
  molar masses, so the reformer sees a slightly lighter feed than the hydrotreater made.

The sulfur table is the other cross-check: the hydrotreaters send about an eighth of the
crude's sulfur to the sulfur plant as H2S, and the rest stays in the residue.

## 9. A gradient through the chain

Every unit returns implicit-function gradients at its converged point, and the pools are
smooth functions of their component streams, so a product-pool quality has an exact
derivative with respect to an upstream operating variable. Here: the ULSD pool's sulfur
and cetane index with respect to the distillate hydrotreater's bed-1 inlet temperature,
through the hydrotreater's recycle tear, its beds and stripper, the jet/diesel split and the
blend pool. One reverse-mode Jacobian (one more compile), checked against a central
difference of two more solves.

In [12]:
def ulsd_quality(T_in):
    r = dht.solve(dht_feed, params=dataclasses.replace(dht_params, T_in=(T_in,)), warn=False)
    _, d = jet_and_diesel(r)
    c = BlendComponent.from_stream("diesel", d, r.product_char, flash_C=60.0)
    p = BlendPool("ulsd")([c], [1.0], basis="split").properties
    return jnp.stack([p["S_ppm"], p["cetane_index"]])


T0 = C + 340.0
t0 = time.time()
J = jax.jacrev(ulsd_quality)(T0)
tick("reverse-mode gradient", t0)
h = 0.5
fd = (ulsd_quality(T0 + h) - ulsd_quality(T0 - h)) / (2 * h)
table([("ULSD sulfur (ppm per K)", float(J[0]), float(fd[0])),
       ("ULSD cetane index (per K)", float(J[1]), float(fd[1]))],
      ["d / d(DHT inlet T)", "implicit (AD)", "central FD, h = 0.5 K"], floatfmt=".5g")

reverse-mode gradient: 229 s


| d / d(DHT inlet T)        |   implicit (AD) |   central FD, h = 0.5 K |
|---------------------------|-----------------|-------------------------|
| ULSD sulfur (ppm per K)   |       -3.5224   |                -3.529   |
| ULSD cetane index (per K) |        0.033838 |                 0.03384 |

Hotter beds take the sulfur down, and they raise the cetane index a little as the aromatics
saturate (until the equilibrium recedes at higher temperature). The two derivatives agree to
the truncation error of the difference. A planner would read these as the hydrotreater's
delta vector (`difflow_refinery.hydrotreating.planning.hdt_block` builds them that way).

The reformer's and the gas plant's gradients work the same way (forward mode for the
reformer, `jax.jacfwd`), but each is another compile of several minutes on this machine,
so they are left to `tests/refinery/test_reforming.py` and `examples/38`.

In [13]:
print(f"whole notebook: {(time.time() - T_START) / 60:.1f} min")

whole notebook: 11.8 min


## 10. Gaps this refinery runs into

Each of these is a place where the notebook above used a stand-in.

1. **Crude unit to gas plant.** No adapter from a crude-unit product to a `gas_components`
   table; the cut selection and folding are example 38's code, repeated. The crude unit
   makes no H2S, so the H2S in the offgas is an assumption.
2. **Hydrotreater product to the gas plant's or the reformer's components.** The treated
   naphtha lives on the hydrotreater's product grid. A rigorous naphtha splitter
   (`gasplant.splitter`) cannot take it without an adapter, and
   `NaphthaFeed.from_characterization` reads the untreated crude's composition, not the
   treated cuts'.
3. **No jet/diesel split after a distillate hydrotreater** (no product fractionator on the
   hydrotreater, no splitter on its product grid). The split is a sharp pseudo-component
   cut at 240 C.
4. **No hydrogen network.** The reformer's net gas and the hydrotreaters' makeup are
   balanced by arithmetic; the makeup purity is the hydrotreaters' default, not the
   reformer's gas.
5. **Product properties not computed:** straight-run octane, flash point, freeze point,
   smoke point, viscosity. All are assumed values above. Sulfur is not carried through the
   reformer.
6. **No residue upgrading or fuel-oil route:** the atmospheric residue is the fuel oil
   as it comes off the crude unit, at a few weight percent sulfur, against a
   0.5 wt% spec. A vacuum unit (`VacuumColumn`) exists; a residue desulfurizer or a
   cutter-stock blend does not.
7. **Hydrotreater furnaces and reporting.** The hydrotreaters have no feed heater (bed-1
   inlet temperature is a spec), so their fired duty is missing from the energy table; and
   their `gas.yield` goes negative on a feed carrying C5 light ends.
8. **Hydrotreater robustness and outputs.** One naphtha condition (320 C, 50 bar,
   LHSV 0.5) ends in a NaN inside a linear solve instead of `converged=False`;
   `product_stream()` drops the light gases dissolved in the liquid products, so a
   downstream mass balance does not close to round-off; and with the illustrative
   constants a naphtha unit at 30 bar dehydrogenates.
9. **No plant-level object.** The connections are Python code, not a `Flowsheet` of the
   units, so a gradient across two units (say the CDU naphtha rate into the reformer's RON)
   has to be written as one function by hand, and each unit compiles separately.